# Phase 1 — Baseline Evaluation
**Model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` (raw base, no fine-tuning)

**Goal:** Establish baseline BLEU and BERTScore on 10 manually crafted test prompts before any fine-tuning.

> **Before running:** Open `test_prompts.json`, paste a GPT-4o / Mistral / DeepSeek response into each `"reference"` field, then upload the file to this Colab session (or mount Google Drive and update `PROMPTS_PATH` below).

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.44.2 bitsandbytes==0.43.3 accelerate==0.34.2 \
    evaluate==0.4.3 bert-score==0.3.13 sacrebleu==2.4.3

In [ ]:
# ── Cell 2: Mount Google Drive (optional — skip if uploading files directly) ──
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import json
import re
import time
import torch
import pandas as pd
import evaluate
from bert_score import score as bert_score_fn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 4: Config ─────────────────────────────────────────────────────────────
MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
PROMPTS_PATH = "test_prompts.json"   # change to Drive path if needed
RESULTS_PATH = "/content/drive/MyDrive/baseline_results.csv"  # where to save
MAX_NEW_TOKENS = 200
TEMPERATURE    = 0.7
TOP_P          = 0.9

In [ ]:
# ── Cell 5: Load test prompts ──────────────────────────────────────────────────
with open(PROMPTS_PATH) as f:
    data = json.load(f)

prompts    = [p["prompt"]    for p in data["prompts"]]
references = [p["reference"] for p in data["prompts"]]
types      = [p["type"]      for p in data["prompts"]]

empty = [i+1 for i, r in enumerate(references) if not r.strip()]
if empty:
    raise ValueError(
        f"Reference answers missing for prompt IDs: {empty}.\n"
        "Please fill in test_prompts.json before running evaluation."
    )

print(f"Loaded {len(prompts)} prompts — all references present.")

In [ ]:
# ── Cell 6: Load model in 4-bit NF4 ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

In [ ]:
# ── Cell 7: Generation helper ──────────────────────────────────────────────────
def format_prompt(instruction: str) -> str:
    """Alpaca-style prompt format."""
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )

@torch.inference_mode()
def generate(instruction: str) -> str:
    prompt = format_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Decode only the newly generated tokens
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
# ── Cell 8: Generate baseline responses ───────────────────────────────────────
print("Generating baseline responses...")
generated = []
for i, prompt in enumerate(prompts):
    t0 = time.time()
    response = generate(prompt)
    elapsed = time.time() - t0
    generated.append(response)
    print(f"[{i+1:02d}/10] ({elapsed:.1f}s) {prompt[:60]}...")
    print(f"   → {response[:120]}...\n")

In [ ]:
# ── Cell 9: Compute BLEU scores ────────────────────────────────────────────────
sacrebleu = evaluate.load("sacrebleu")

bleu_scores = []
for gen, ref in zip(generated, references):
    result = sacrebleu.compute(
        predictions=[gen],
        references=[[ref]],
    )
    bleu_scores.append(round(result["score"], 4))

print("BLEU scores:", bleu_scores)
print(f"Mean BLEU: {sum(bleu_scores)/len(bleu_scores):.4f}")

In [ ]:
# ── Cell 10: Compute BERTScore ─────────────────────────────────────────────────
P, R, F1 = bert_score_fn(
    generated,
    references,
    lang="en",
    rescale_with_baseline=True,
    verbose=True,
)

bert_f1 = [round(f.item(), 4) for f in F1]
bert_p  = [round(p.item(), 4) for p in P]
bert_r  = [round(r.item(), 4) for r in R]

print("BERTScore F1:", bert_f1)
print(f"Mean BERTScore F1: {sum(bert_f1)/len(bert_f1):.4f}")

In [ ]:
# ── Cell 11: Results table ─────────────────────────────────────────────────────
df = pd.DataFrame({
    "id":           range(1, 11),
    "type":         types,
    "prompt":       prompts,
    "reference":    references,
    "generated":    generated,
    "bleu":         bleu_scores,
    "bert_p":       bert_p,
    "bert_r":       bert_r,
    "bert_f1":      bert_f1,
})

pd.set_option("display.max_colwidth", 80)
print(df[["id","type","bleu","bert_f1"]].to_string(index=False))
print(f"\nMean BLEU:         {df['bleu'].mean():.4f}")
print(f"Mean BERTScore F1: {df['bert_f1'].mean():.4f}")

In [ ]:
# ── Cell 12: Save results to Google Drive ─────────────────────────────────────
df.to_csv(RESULTS_PATH, index=False)
print(f"Results saved to {RESULTS_PATH}")

# Also print full generated outputs for report
print("\n" + "="*80)
print("FULL GENERATED OUTPUTS (for report)")
print("="*80)
for i, row in df.iterrows():
    print(f"\n[{row['id']}] {row['type'].upper()}: {row['prompt']}")
    print(f"Generated: {row['generated']}")
    print(f"BLEU: {row['bleu']}  |  BERTScore F1: {row['bert_f1']}")

## Summary

Copy the mean BLEU and mean BERTScore F1 into **Section 3 (Baseline Results)** of `report.md`.

**Next step:** Run `01_sft_training.ipynb` to fine-tune TinyLlama on `tatsu-lab/alpaca`.